In [17]:
import os
import sys
module_path = os.path.abspath(os.path.join(r'C:\Users\Adriano Matos\Documents\Python Scripts\big-raster-helper\src'))
container_file_path = '/credentials/robust-raster-cefaedc5282c.json'  # Path inside the container

if module_path not in sys.path:
    sys.path.append(module_path)

In [3]:
import math
import pandas as pd
import psutil

# Read the CSV file into a DataFrame
df = pd.read_csv('metrics_report.csv')

def get_available_system_memory():
    # Get total available RAM in bytes
    total_ram = psutil.virtual_memory().total

    # Convert from bytes to gigabytes (GB)
    total_ram_gb = total_ram / (1024 ** 3)

    return total_ram_gb

# Get the max workers (RAM limited)
available_system_memory = get_available_system_memory()
ram_safety_threshold = 0.5
max_workers_ram_limited = int(df['wRAM'][0])
memory_per_worker = available_system_memory * ram_safety_threshold / max_workers_ram_limited
print(df['wRAM'][0])
print(available_system_memory)
print(memory_per_worker)
#dcm.stop_and_remove_containers()

ee_key_host_path =r'C:\Users\Adriano Matos\Documents\credentials\robust-raster-cefaedc5282c.json' # Path in the host machine
ee_key_container_path = '/credentials/robust-raster-cefaedc5282c.json'  # Path inside the container

nc_host_path = r'C:\Users\Adriano Matos\Documents\Python Scripts\big-raster-helper\experiment_code\saved_on_disk.nc'
nc_container_path = '/data/saved_on_disk.nc'

volumes = {
    ee_key_host_path: {'bind': ee_key_container_path, 'mode': 'ro'},  # 'ro' means read-only
    nc_host_path: {'bind': nc_container_path, 'mode': 'ro'},  # 'ro' means read-only
}

from dask_cluster_manager import DaskClusterManager
dcm = DaskClusterManager()
dcm.create_cluster(num_workers=max_workers_ram_limited, n_threads=1, memory_limit=f"{memory_per_worker}GB", volumes=volumes)
#dcm.create_cluster(num_workers=int(df['wRAM'][0]), n_threads=1, memory_limit=f"{memory_per_worker}GB", volumes=volumes)

# I need to have a condition that chunks EE Data if I am doing a test or if I am doing a full run
# If test, then chunk to the size of the dataset
# If not, then chunk to 48 MBs?
# We might be limited to 48MBs in Earth Engine...

16
31.767879486083984
0.9927462339401245
Dask Scheduler started with ID 139c803f0542b6a8f34593fdd615909abedd222a6f5a0a4b71898747656545ee
Dask Worker 1 started with ID 99645a2caed0eba16c3e77e2a240bcf784d9d1567a136659abe4e94b93c1aec6
Dask Worker 2 started with ID d7dcdea7bd2f4e359f6fea4f3a9dc310bb70921d8a9e5cc3db953e39e8f4a448
Dask Worker 3 started with ID cd1162bd6ec4a1740ce04aa083403c0e797fe83662f20367d80defcc65e9099d
Dask Worker 4 started with ID 44e3191d90553adac66e087b636dc862aa88009198b491acc4886cdd5b20874b
Dask Worker 5 started with ID 2a260cd84fede0eca48cbf97bb6a311e637311eea29a261631396819690ed3b1
Dask Worker 6 started with ID 80c04ab840a0084c234bf67e69e165c8896423577f07b7c0ede8cc78959f37a9
Dask Worker 7 started with ID 3c5c2404b96200c50e29bb78454b4696713da684092bd1887b35ea71cee65938
Dask Worker 8 started with ID 9b073f156a80b7f9acbd3b603612fdcfc763ddba19ade629eb6f7a753cd5cc1b
Dask Worker 9 started with ID 04495dcefcb007dfb97e27203409bad7bb4b550ff895e3538ced1e337266b9ba
Dask Wor

In [ ]:
# Set up Dask cluster using Docker Python SDK
import docker
import pandas as pd
import psutil

# Initialize Docker client
docker_client = docker.from_env()

# Create a Docker network
network_name = "dask-network"

try:
    docker_client.networks.get(network_name)
except docker.errors.NotFound:
    docker_client.networks.create(network_name, driver="bridge")

# Define paths for volume mounting
host_file_path = r'C:\Users\Adriano Matos\Documents\credentials\robust-raster-cefaedc5282c.json'  # Host machine path
container_file_path = '/credentials/robust-raster-cefaedc5282c.json'  # Path inside the container

host_nc_path = r'C:\Users\Adriano Matos\Documents\Python Scripts\big-raster-helper\experiment_code\saved_on_disk.nc'
container_nc_path = '/data/saved_on_disk.nc'

# Volume mapping
volumes = {
    host_file_path: {'bind': container_file_path, 'mode': 'ro'},  # 'ro' means read-only
    host_nc_path: {'bind': container_nc_path, 'mode': 'ro'},
}

volumes = {
    "c:/Users/Adriano Matos/Documents/Python Scripts/big-raster-helper/experiment_code": {
        "bind": "/experiment_code",
        "mode": "rw"
    },
    host_nc_path: {  # replace `host_nc_path` with the actual path
        "bind": container_nc_path,  # replace `container_nc_path` with the actual container path
        "mode": "rw"
    },
    host_file_path: {'bind': container_file_path, 'mode': 'ro'}
}


# Run Dask Scheduler
scheduler = docker_client.containers.run(
    "adrianomdocker/rrtest",
    command="dask-scheduler",
    name="dask-scheduler",
    network=network_name,
    detach=True,
    ports={'8786/tcp': 8786, '8787/tcp': 8787},
)

print(f"Dask Scheduler started with ID {scheduler.id}")

# Run Dask Workers
# Read the CSV file into a DataFrame
df = pd.read_csv('metrics_report.csv')

def get_available_system_memory():
    # Get total available RAM in bytes
    total_ram = psutil.virtual_memory().total

    # Convert from bytes to gigabytes (GB)
    total_ram_gb = total_ram / (1024 ** 3)

    return total_ram_gb

# Get the max workers (RAM limited)
available_system_memory = get_available_system_memory()
ram_safety_threshold = 0.5
max_workers_ram_limited = int(df['wRAM'][0])
memory_per_worker = available_system_memory * ram_safety_threshold / max_workers_ram_limited
memory_per_worker = f"{memory_per_worker}GB"

num_workers = 16  # Specify the number of workers you want
n_threads = 1
container_mem_limit = "16GB"
dask_mem_limit = "8GB"
memory_per_worker = "2GB"

worker_ids = []
for i in range(num_workers):
    worker = docker_client.containers.run(
        "adrianomdocker/rrtest",
        command=f"dask-worker dask-scheduler:8786 --nthreads {n_threads} --memory-limit {memory_per_worker}",
        name=f"dask-worker-{i+1}",
        network=network_name,
        detach=True,
        mem_limit=memory_per_worker,
        volumes=volumes
)
    worker_ids.append(worker.id)
    print(f"Dask Worker {i+1} started with ID {worker.id}")

# Connect to the Dask Scheduler
#dask_client = Client('tcp://localhost:8786')

print("Connected to Dask Scheduler")

# Monitor the Dask Dashboard
print("Dask dashboard available at http://localhost:8787")

# The cluster is now set up and can be used within the script or interactively.

Dask Scheduler started with ID d1972952219a13346b559d9b94c10f09112a2ef07b54ec972ea4ecee7018660b
Dask Worker 1 started with ID 0f784c268f6f4453662b0ee69a2bb9fd5f693ad11f9dd592b2d8266dd34c0945
Dask Worker 2 started with ID e9ec270e1c21009816aec1e54d89caf5596723c9bcb1da4c7b6bd9f18571071a
Dask Worker 3 started with ID e8ab1faebcc4a3191b65763966c8675c9ac5b6c9d4f6efbfec067bb80e02ef00
Dask Worker 4 started with ID 4f7a447abbb4f6381bd868c0e54f114136a2eed839bef8a3910e3c6254fef638
Dask Worker 5 started with ID d61dbf4dddb1945f62d192992eede7a955351da86cddcb6991a2c1ea2a234b85
Dask Worker 6 started with ID e8baa838fe4d859838877c9611ba4402f15845af1ca7d32806410df1ad94f4ba
Dask Worker 7 started with ID 12a4000c2e38f476dbce92fe64b8619263fd6172d55877f122c19457caf2c29c
Dask Worker 8 started with ID 14126af58c09ee3145b1d030deabf3191451b27d45fbc6924cb650238665fc6d
Dask Worker 9 started with ID c4bf6c1c8ec42ee2638302e1eb0a3fe4460bfc91ad80924d19c7b8aaa5e00d43
Dask Worker 10 started with ID 3f90ae95cc1c5322bd

In [3]:
print(type(docker_client))

<class 'docker.client.DockerClient'>


In [19]:
from dask.distributed import Client
dask_client = Client('tcp://localhost:8786')

In [11]:
from dask_plugins import EEPlugin

ee_plugin = EEPlugin(container_file_path)
dask_client.register_plugin(ee_plugin)

{'tcp://172.20.0.3:33869': {'status': 'OK'},
 'tcp://172.20.0.4:36193': {'status': 'OK'}}

In [13]:
# EE credentials with a JSON key
import json
import ee
json_key = r'C:\Users\Adriano Matos\Documents\credentials\robust-raster-cefaedc5282c.json'
with open(json_key, 'r') as file:
    data = json.load(file)
credentials = ee.ServiceAccountCredentials(data["client_email"], json_key)

In [ ]:
###############################
### TESTING MY PACKAGE CODE ###
###############################

In [14]:
# Earth Engine xarray
import sys
import os
import ee
import os

#ee.Initialize(opt_url='https://earthengine-highvolume.googleapis.com', project='robust-raster')
ee.Initialize(credentials = credentials, opt_url='https://earthengine-highvolume.googleapis.com')

def prep_sr_l8(image):
    # Develop masks for unwanted pixels (fill, cloud, cloud shadow).
    qa_mask = image.select('QA_PIXEL').bitwiseAnd(int('11111', 2)).eq(0)
    saturation_mask = image.select('QA_RADSAT').eq(0)

    # Apply the scaling factors to the appropriate bands.
    def get_factor_img(factor_names):
        factor_list = image.toDictionary().select(factor_names).values()
        return ee.Image.constant(factor_list)
    
    scale_img = get_factor_img([
        'REFLECTANCE_MULT_BAND_.|TEMPERATURE_MULT_BAND_ST_B10'])
    offset_img = get_factor_img([
        'REFLECTANCE_ADD_BAND_.|TEMPERATURE_ADD_BAND_ST_B10'])
    scaled = image.select('SR_B.|ST_B10').multiply(scale_img).add(offset_img)

    # Replace original bands with scaled bands and apply masks.
    return image.addBands(scaled, None, True)\
        .updateMask(qa_mask).updateMask(saturation_mask)

if module_path not in sys.path:
    sys.path.append(module_path)

from input_driver import EarthEngineDataset

CALIFORNIA = ee.FeatureCollection("projects/calfuels/assets/Boundaries/California")
LTBMU = ee.FeatureCollection("projects/calfuels/assets/Boundaries/LTBMU_remove_NV_remove_lake")

chunk_size  = {
            'time': 48,
            'X': 512,
            'Y': 256
}

parameters = {
            'collection': 'LANDSAT/LC08/C02/T1_L2',
            'bands': ['SR_B4', 'SR_B5'],
            'start_date': '2020-12-29',
            'end_date': '2020-12-31',
            'geometry': CALIFORNIA.geometry(),
            'crs': 'EPSG:3310',
            'scale': 100,
            'chunks': chunk_size,
            'map_function': prep_sr_l8
        }

landsat_xarray = EarthEngineDataset(parameters)

In [1]:
# Template xarray based on Earth Engine
import numpy as np
import pandas as pd
import xarray as xr

# Define the dimensions
time = pd.date_range("2020-12-29T18:57:32.281000", periods=3)
X = np.linspace(-421600, 486700, 9084)
Y = np.linspace(-599200, 458500, 10578)

# Create a data array with random data for each variable
data = np.random.rand(len(time), len(X), len(Y)).astype(np.float32)

# Create a dictionary of data variables
data_vars = {
    #'SR_B1': (['time', 'X', 'Y'], data),
    #'SR_B2': (['time', 'X', 'Y'], data),
    #'SR_B3': (['time', 'X', 'Y'], data),
    'SR_B4': (['time', 'X', 'Y'], data),
    'SR_B5': (['time', 'X', 'Y'], data),
    #'SR_B6': (['time', 'X', 'Y'], data),
    #'ST_EMSD': (['time', 'X', 'Y'], data),
    #'ST_QA': (['time', 'X', 'Y'], data),
    #'ST_TRAD': (['time', 'X', 'Y'], data),
    #'ST_URAD': (['time', 'X', 'Y'], data),
    #'QA_PIXEL': (['time', 'X', 'Y'], data),
    #'QA_RADSAT': (['time', 'X', 'Y'], data),
}

chunk_size = {'time': 3, 'X': 512, 'Y': 256}

# Create the dataset
ds = xr.Dataset(
    data_vars=data_vars,
    coords={'time': time, 'X': X, 'Y': Y},
    attrs={
        'date_range': '[1365638400000, 1654560000000]',
        'description': '<p>This dataset contains atmospherically corrected data.</p>',
        'keywords': ['cfmask', 'cloud', 'fmask', 'global', 'l8sr', 'landsat'],
        'period': 0,
        'visualization_2_max': 30000.0,
        'visualization_2_min': 0.0,
        'visualization_2_name': 'Shortwave Infrared (753)',
        'crs': 'EPSG:3310'
    }
).chunk(chunk_size)


In [ ]:
# Compute NDVI on an xarray
import dask.array as da
import xarray as xr
from dask.distributed import performance_report

def compute_ndvi(dataset):
    """
    Compute NDVI from an xarray Dataset using Dask.
    
    Parameters:
    dataset (xarray.Dataset): The input dataset containing NIR and Red bands.
    
    Returns:
    xarray.DataArray: NDVI values computed from the input dataset.
    """
    # Access the NIR (SR_B5) and Red (SR_B4) bands
    nir = dataset['SR_B5']
    red = dataset['SR_B4']
    
    # Ensure the arrays are dask arrays
    nir = nir.data if isinstance(nir, xr.DataArray) else nir
    red = red.data if isinstance(red, xr.DataArray) else red
    
    # Compute NDVI using Dask array operations
    ndvi = (nir - red) / (nir + red)
    
    # Convert the resulting dask array back to a DataArray
    ndvi_da = xr.DataArray(ndvi, dims=dataset['SR_B5'].dims, coords=dataset['SR_B5'].coords, name='NDVI')
    dataset['NDVI'] = ndvi_da
    
    return dataset

# Example usage with the dataset created earlier
ndvi = compute_ndvi(dataset)
ndvi.compute()
#with performance_report(filename='dask-report-nodf.html'):
#    ndvi.compute()

In [3]:
# Sample template dataset
import xarray as xr
import numpy as np
import dask.array as da

chunk_size = ds['SR_B4'].chunks

# Create a template with dask arrays using the same chunk sizes and attributes
template = xr.Dataset(
    {
        'SR_B4': (['time', 'X', 'Y'], da.empty((3, 9084, 10578), chunks=chunk_size, dtype=np.float32)),
        'SR_B5': (['time', 'X', 'Y'], da.empty((3, 9084, 10578), chunks=chunk_size, dtype=np.float32)),
        #'ndvi':  (['time', 'X', 'Y'], da.empty((3, 9084, 10578), chunks=chunk_size, dtype=np.float32))
    },
    coords={
        'time': ds.coords['time'],
        'X': ds.coords['X'],
        'Y': ds.coords['Y'],
    },
    attrs=ds.attrs  # Copy over the original dataset's attributes
)

In [26]:
# Create a function to generate the template based on the original dataset
def create_template(ds, process_func):
    # Extract a single chunk to determine the output structure
    one_chunk = ds.isel(time=slice(0, ds.chunks['time'][0]), 
                        X=slice(0, ds.chunks['X'][0]), 
                        Y=slice(0, ds.chunks['Y'][0]))
    
    # Apply the processing function to this chunk
    processed_chunk = process_func(one_chunk)
    
    # Create the template using a combination of original data variables and newly created ones
    template_vars = {}
    
    for var in processed_chunk.data_vars:
        if var in ds.data_vars:
            # Use the original dataset's shape and chunking for existing variables
            template_vars[var] = (processed_chunk[var].dims, 
                                  da.empty(ds[var].shape, 
                                           chunks=ds[var].chunks, 
                                           dtype=processed_chunk[var].dtype))
        else:
            # For new variables, define the shape and chunks manually based on the original chunking strategy
            new_var_shape = tuple(ds.dims[dim] for dim in processed_chunk[var].dims)
            new_var_chunks = tuple(ds.chunks[dim][0] for dim in processed_chunk[var].dims)
            template_vars[var] = (processed_chunk[var].dims, 
                                  da.empty(new_var_shape, 
                                           chunks=new_var_chunks, 
                                           dtype=processed_chunk[var].dtype))
    
    template = xr.Dataset(
        template_vars,
        coords={coord: ds.coords[coord] for coord in ds.coords},
        attrs=ds.attrs
    )
    
    return template

# Assuming `ds` is your chunked xarray object:
template = create_template(ds, process_chunk_as_pandas)

In [20]:
# Load the data using Dask Delayed!
import xarray as xr
import dask

chunk_size  = {
            'time': 3,
            'X': 2048,
            'Y': 2048
}

# Define a delayed function that opens the dataset
@dask.delayed
def load_dataset_on_worker():
    return xr.open_dataset('/data/saved_on_disk.nc', chunks=chunk_size)

# Trigger the delayed execution, this will happen on the worker
delayed_ds = load_dataset_on_worker()

# Optionally, you can convert the delayed object into a Dask-backed xarray Dataset
ds = delayed_ds.compute()

# Now ds can be used for further computation on the Dask workers

In [18]:
print(ds)

<xarray.Dataset> Size: 2GB
Dimensions:  (time: 3, X: 9084, Y: 10578)
Coordinates:
  * time     (time) datetime64[ns] 24B 2020-12-29T18:57:32.281000 ... 2020-12...
  * X        (X) float64 73kB -4.216e+05 -4.215e+05 ... 4.866e+05 4.867e+05
  * Y        (Y) float64 85kB -5.992e+05 -5.991e+05 ... 4.584e+05 4.585e+05
Data variables:
    SR_B4    (time, X, Y) float32 1GB dask.array<chunksize=(3, 512, 256), meta=np.ndarray>
    SR_B5    (time, X, Y) float32 1GB dask.array<chunksize=(3, 512, 256), meta=np.ndarray>
Attributes:
    date_range:            [1365638400000, 1654560000000]
    description:           <p>This dataset contains atmospherically corrected...
    keywords:              ['cfmask', 'cloud', 'fmask', 'global', 'l8sr', 'la...
    period:                0
    visualization_2_max:   30000.0
    visualization_2_min:   0.0
    visualization_2_name:  Shortwave Infrared (753)
    crs:                   EPSG:3310


In [21]:
import xarray as xr
import pandas as pd

def process_chunk_as_pandas(chunk):
    # Convert xarray chunk to pandas DataFrame
    df = chunk.to_dataframe().reset_index()
    
    # Perform your calculations
    df['ndvi'] = df['SR_B4'] + df['SR_B5']
    
    # Convert the pandas DataFrame back to an xarray structure
    df = df.set_index(list(chunk.dims))
    result = df.to_xarray()
    
    return result

# Assuming `your_chunked_xarray` is your chunked xarray object:
result = xr.map_blocks(process_chunk_as_pandas, ds)

In [25]:
# Write results to a NetCDF file
result.to_netcdf("/data/processed_data.nc")
# Trigger computation

PermissionError: [Errno 13] Permission denied: 'c:\\data\\processed_data.nc'

In [ ]:
result.compute()

In [ ]:
import rioxarray
result["NDVI"].isel(time=0).rio.to_raster("ndvi.tif")

In [21]:
print(ds)

<xarray.Dataset> Size: 2GB
Dimensions:  (time: 3, X: 9084, Y: 10578)
Coordinates:
  * time     (time) datetime64[ns] 24B 2020-12-29T18:57:32.281000 ... 2020-12...
  * X        (X) float64 73kB -4.216e+05 -4.215e+05 ... 4.866e+05 4.867e+05
  * Y        (Y) float64 85kB -5.992e+05 -5.991e+05 ... 4.584e+05 4.585e+05
Data variables:
    SR_B4    (time, X, Y) float32 1GB dask.array<chunksize=(3, 512, 256), meta=np.ndarray>
    SR_B5    (time, X, Y) float32 1GB dask.array<chunksize=(3, 512, 256), meta=np.ndarray>
Attributes:
    date_range:            [1365638400000, 1654560000000]
    description:           <p>This dataset contains atmospherically corrected...
    keywords:              ['cfmask', 'cloud', 'fmask', 'global', 'l8sr', 'la...
    period:                0
    visualization_2_max:   30000.0
    visualization_2_min:   0.0
    visualization_2_name:  Shortwave Infrared (753)
    crs:                   EPSG:3310


In [ ]:
import xarray as xr
import pandas as pd
import sys, os

module_path = os.path.abspath(os.path.join(r'C:\Users\Adriano Matos\Documents\Python Scripts\big-raster-helper\src'))

if module_path not in sys.path:
    sys.path.append(module_path)

def process_chunk_as_pandas(df):
    # Perform your calculations
    df['ndvi'] = df['SR_B4'] + df['SR_B5']
    
    return df

from udf_tuner import UserDefinedFunction

user_defined_func = UserDefinedFunction()

# Apply the user-defined function
result = user_defined_func.apply_user_function(ds, process_chunk_as_pandas)

In [18]:
import xarray as xr
import pandas as pd

def process_chunk_as_pandas(df):
    # Perform your calculations
    df['ndvi'] = df['SR_B4'] + df['SR_B5']
    
    return df

In [22]:
from udf_tuner import UserDefinedFunction

user_defined_func = UserDefinedFunction()

# Apply the user-defined function
result = user_defined_func.apply_user_function(ds, process_chunk_as_pandas)

In [23]:
result.compute()

KeyboardInterrupt: 